In [ ]:
from pathlib import Path

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split


In [ ]:
DATA_PATH = Path("../data/raw/seg_train_test")

IMG_SIZE = (150, 150)
BATCH_SIZE = 32
SEED = 42


In [ ]:
image_paths = []
labels = []

class_names = sorted([
    folder.name
    for folder in DATA_PATH.iterdir()
    if folder.is_dir()
])

for label, class_name in enumerate(class_names):
    class_path = DATA_PATH / class_name

    for image_path in class_path.glob("*.jpg"):
        image_paths.append(str(image_path))
        labels.append(label)

print(class_names)
print("Total images:", len(image_paths))


In [ ]:
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    image_paths,
    labels,
    test_size=0.2,
    random_state=SEED,
    stratify=labels
)

val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths,
    temp_labels,
    test_size=0.5,
    random_state=SEED,
    stratify=temp_labels
)

print("Train:", len(train_paths))
print("Validation:", len(val_paths))
print("Test:", len(test_paths))


In [ ]:
def load_image(path, label):
    image = tf.io.read_file(path)

    image = tf.image.decode_jpeg(
        image,
        channels=3
    )

    image = tf.image.resize(
        image,
        IMG_SIZE
    )

    return image, label


In [ ]:
test_ds = tf.data.Dataset.from_tensor_slices(
    (
        test_paths,
        test_labels
    )
)

test_ds = test_ds.map(
    load_image
)

test_ds = test_ds.batch(
    BATCH_SIZE
)


In [ ]:
model = tf.keras.models.load_model(
    "../models/best_model.keras",
    compile=False
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
